In [48]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import pandas as pd

In [49]:
import random

def random_flip(image):
    if random.choice([True, False]):
        image = np.fliplr(image)
    return image

def add_salt_and_pepper_noise(image, salt_prob=0.01, pepper_prob=0.01):
    noisy_image = image.copy()
    total_pixels = image.size

    num_salt = int(salt_prob * total_pixels)
    salt_coords = [np.random.randint(0, i - 1, num_salt) for i in image.shape]
    noisy_image[salt_coords[0], salt_coords[1]] = 255

    num_pepper = int(pepper_prob * total_pixels)
    pepper_coords = [np.random.randint(0, i - 1, num_pepper) for i in image.shape]
    noisy_image[pepper_coords[0], pepper_coords[1]] = 0

    return noisy_image


def normalize_depth(depth_img):
    depth_min = np.min(depth_img)
    depth_max = np.max(depth_img)
    normalized_depth = (depth_img - depth_min) / (depth_max - depth_min) * 255
    return normalized_depth.astype(np.uint8)


def enhance_depth_contrast(depth_img, gamma=2.0):
    normalized_depth = normalize_depth(depth_img)
    enhanced_depth = np.power(normalized_depth / 255.0, gamma) * 255
    return np.clip(enhanced_depth, 0, 255).astype(np.uint8)


def resize_w_enhanced_depth(image, target_size=(256, 384), gamma=1.2):
    resized_image = cv.resize(
        image, target_size, interpolation=cv.INTER_LINEAR
    )
    return resized_image

In [50]:
total = {}
AUGMENT = False
LIM = 200
for batch in os.listdir("batches"):
    batch_path = os.path.join("batches", batch)
    if os.path.isdir(batch_path):
        for item in tqdm(os.listdir(batch_path)):
            df = []
            item_path = os.path.join(batch_path, item)
            if os.path.isdir(item_path):
                c = 0
                for img in os.listdir(item_path):
                    img_path = os.path.join(item_path, img)
                    if os.path.isfile(img_path) and "depth" in img_path:
                        image = Image.open(img_path)
                        image = np.array(image)
                        image[image <= 1] = 0
                        df.append(image)
                        c += 1
                    if c == LIM:
                        break
            total[item_path] = {"data": df}

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00,  7.47it/s]


In [51]:
reference = 1280 * 480

for key, df in total.items():
    pixel_sizes = [x.shape[0] * x.shape[1] for x in df["data"]]
    
    # Binarize non-zero pixels to 1s, then count
    nonzero_pixel_counts = [np.count_nonzero(x > 0) for x in df["data"]]
    std_depth = [np.std(x) for x in df["data"]]
    approx_volume = [np.sum(x) for x in df["data"]]
    aspect_ratios = [x.shape[1] / x.shape[0] for x in df["data"]]
    mean_depth = [np.mean(x[x > 0]) if np.count_nonzero(x > 0) > 0 else 0 for x in df["data"]]

    def minmax(arr):
        arr = np.array(arr)
        return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

    total[key]["non_zero_pixels"] = minmax(nonzero_pixel_counts)
    total[key]["pixel_sizes"] = minmax(pixel_sizes)
    
    raw_ratio = np.array(nonzero_pixel_counts) / np.array(pixel_sizes)
    total[key]["pixel_to_non_ratio"] = minmax(raw_ratio)

    total[key]["std_depth"] = minmax(std_depth)
    total[key]["mean_depth"] = minmax(mean_depth)
    total[key]["pixel_ratio"] = minmax([size / reference for size in nonzero_pixel_counts])
    total[key]["volume_proxy"] = minmax(approx_volume)
    total[key]["aspect_ratio"] = minmax(aspect_ratios)

    adjusted_df = [resize_w_enhanced_depth(x) for x in df["data"]]
    total[key]["data"] = adjusted_df

In [52]:
weights = pd.read_csv("weight.csv")
weights, weights.shape

(   pig_num     weight
 0        1  15.218280
 1        3  17.328257
 2        4  17.019110
 3        5  16.761245
 4        8  15.987649,
 (5, 2))

In [65]:
weights

array([17.328257], dtype=float32)

In [54]:
for (key, df), w in zip(total.items(), weights['weight'].to_list()):
    total[key]["weight"] = w

In [55]:
summary_stats = {}

for key, df in total.items():
    summary = {}
    for feature in ["pixel_to_non_ratio", "non_zero_pixels", "pixel_sizes", "std_depth", "pixel_ratio", "volume_proxy", "aspect_ratio"]:
        values = df.get(feature)
        if isinstance(values, np.ndarray):
            summary[feature + "_mean"] = np.mean(values)
    summary_stats[key] = summary

summary_stats

{'batches\\1\\pig_A': {'pixel_to_non_ratio_mean': 0.43042578914776886,
  'non_zero_pixels_mean': 0.6618593991815939,
  'pixel_sizes_mean': 0.7158728176868288,
  'std_depth_mean': 0.6446373632872546,
  'pixel_ratio_mean': 0.661859271957718,
  'volume_proxy_mean': 0.6639366105631588,
  'aspect_ratio_mean': 0.20719834917547952},
 'batches\\1\\pig_B': {'pixel_to_non_ratio_mean': 0.326886713202958,
  'non_zero_pixels_mean': 0.21221876188654804,
  'pixel_sizes_mean': 0.41177678571405585,
  'std_depth_mean': 0.5181856985487151,
  'pixel_ratio_mean': 0.21221869989198883,
  'volume_proxy_mean': 0.21464334937401627,
  'aspect_ratio_mean': 0.44509957779970577},
 'batches\\1\\pig_C': {'pixel_to_non_ratio_mean': 0.5573114473243062,
  'non_zero_pixels_mean': 0.42734409247146,
  'pixel_sizes_mean': 0.3508299984952523,
  'std_depth_mean': 0.6455683586460795,
  'pixel_ratio_mean': 0.42734395768721956,
  'volume_proxy_mean': 0.42665395756169483,
  'aspect_ratio_mean': 0.32329143386281406},
 'batches\\1\

In [56]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

class DepthImageDataset(Dataset):
    def __init__(self, data_dict):
        self.images = []
        self.features = []
        self.weights = []

        for key in data_dict.keys():
            images = data_dict[key]["data"]
            feature_keys = [k for k in data_dict[key].keys() if k not in ['data', 'weight']]

            # Collect per-image feature vectors
            per_image_features = np.stack([data_dict[key][k] for k in feature_keys], axis=1)

            # Same weight for all images in this group
            w = [data_dict[key]["weight"]] * len(images)

            self.images.extend(images)
            self.features.extend(per_image_features)
            self.weights.extend(w)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        feature = torch.tensor(self.features[idx], dtype=torch.float32)
        weight = torch.tensor(self.weights[idx], dtype=torch.float32)
        return image, feature, weight

In [57]:
# Change the number of keys to fit what you have. In this example, 2 pigs will be used to train, and the last 2 will be used for validation.
train_val_ratio = 0.7 # 10 pigs = 8 train, 2 val
all_keys = list(total.keys())
random.shuffle(all_keys)

num_total = len(all_keys)
num_train = int(num_total * train_val_ratio)

train_keys = all_keys[:num_train]
val_keys = all_keys[num_train:]

train_data_dict = {key: total[key] for key in train_keys}
val_data_dict = {key: total[key] for key in val_keys}

In [58]:
{k:v for k,v in zip(train_keys, weights['weight'][:num_train])}, {k:v for k,v in zip(val_keys, weights['weight'][num_train:])}

({'batches\\1\\pig_E': 15.21828,
  'batches\\1\\pig_B': 17.3282571,
  'batches\\1\\pig_D': 17.0191098},
 {'batches\\1\\pig_C': 16.7612445, 'batches\\1\\pig_A': 15.9876486})

In [59]:
train_dataset = DepthImageDataset(train_data_dict)
val_dataset = DepthImageDataset(val_data_dict)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=True)

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split

X = []
y = []

for batch in train_dataloader:
    features = batch[1].detach().cpu().numpy() 
    w = batch[-1].detach().cpu().numpy()

    X.extend(features.reshape(features.shape[0], -1))
    y.extend(w.flatten())

X = np.array(X)
y = np.array(y)

In [61]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

lgbm = LGBMRegressor(verbose = -1)
lgbm.fit(X_train, y_train)

preds = lgbm.predict(X_val)

In [62]:
class CNN(nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels=1,
    ):
        super(CNN, self).__init__()
        # You can not change the conv1(in_channels) and fc4 (out_features).
        # Do not change kernel_size, stride, padding.
        # You can change the out channels but make sure to adjust for the consequent layer.
        # Example: conv1(out_channels) is 16 so conv2(in_channels) must be 16 or conv1(out_channels) = conv2(in_channels).

        # Image Encoder
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=hidden_channels,
            kernel_size=3,
            stride=1,
            padding=1,
        )

        self.conv2 = nn.Conv2d(
            in_channels=hidden_channels,
            out_channels=hidden_channels * 2,
            kernel_size=3,
            stride=1,
            padding=1,
        )

        self.conv3 = nn.Conv2d(
            in_channels=hidden_channels * 2,
            out_channels=hidden_channels * 4,
            kernel_size=3,
            stride=1,
            padding=1,
        )

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)
        self.fc1 = nn.Linear(264 * hidden_channels, hidden_channels * 4)
        self.fc2 = nn.Linear(hidden_channels * 4, hidden_channels * 2)
        self.fc3 = nn.Linear(hidden_channels * 2 + 1, out_channels)

    def forward(self, x, x_pred):
        
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))

        x = x.reshape(x.size(0), -1)

        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        if x_pred.dim() == 1:
            x_pred = x_pred.unsqueeze(1)
        x = torch.cat([x, x_pred], dim=1)
        x = torch.relu(self.fc3(x))
        return x

In [63]:
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

In [64]:
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN(in_channels = 1, hidden_channels = 32)
model = model.to(device=device)

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.0002)

# If you want to add or remove features you must also change the size of this.
num_epochs = 10
total_losses = []

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

for epoch in range(num_epochs):

    model.train()

    losses = {"train": 0.0, "val": 0.0}

    for images, features, w in train_dataloader:
        images = images.to(device=device)
        features = features.cpu().numpy()
        
        if images.shape[1] != 1:  # Check if it's not grayscale
            images = images[:, 0, :, :].unsqueeze(1)

        x_preds = lgbm.predict(np.array(features))
        x_preds = torch.tensor(x_preds, dtype=torch.float32).to(images.device)
        w = w.to(device=device)
        optimizer.zero_grad()

        outputs = model(images, x_preds)

        loss = criterion(outputs.squeeze(), w)

        loss.backward()
        optimizer.step()

        losses["train"] += loss.item()

    model.eval()

    for images, features, w in val_dataloader:
        images = images.to(device=device)
        features = features.cpu().numpy()

        if images.shape[1] != 1:
            images = images[:, 0, :, :].unsqueeze(1)
        
        w = w.to(device=device)
        x_preds = lgbm.predict(np.array(features))
        x_preds = torch.tensor(x_preds, dtype=torch.float32).to(images.device)

        outputs = model(images, x_preds)

        loss = criterion(outputs.squeeze(), w)
        losses["val"] += loss.item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {losses['train'] / len(train_dataloader)}, Evaluation Loss: {losses['val'] / len(val_dataloader)}"
    )

    total_losses.append(losses)

train_losses = [loss["train"] / len(train_dataloader) for loss in total_losses]
val_losses = [loss["val"] / len(val_dataloader) for loss in total_losses]

warmup = 1
plt.figure(figsize=(10, 6))
plt.plot(
    range(warmup + 1, num_epochs + 1), train_losses[warmup:], label="Train Loss", marker="o"
)
plt.plot(
    range(warmup + 1, num_epochs + 1),
    val_losses[warmup:],
    label="Validation Loss",
    marker="s",
)
plt.title("Training and Validation Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Total parameters: 1,182,466
Trainable parameters: 1,182,466


KeyboardInterrupt: 

In [ ]:
import time
torch.save(model.state_dict(), f"pt//model_{time.time()}.pt")

In [ ]:
model.eval()
device = "cuda"

val_rmse = 0
num_batches = 0

with torch.no_grad():
    for images, features, labels in val_dataloader:
        images = images.to(device=device)
        features = features.cpu().numpy()

        if images.shape[1] != 1:
            images = images[:, 0, :, :].unsqueeze(1)
        
        x_preds = lgbm.predict(np.array(features))
        x_preds = torch.tensor(x_preds, dtype=torch.float32).to(images.device)

        outputs = model(images, x_preds)

        mse = nn.MSELoss()(outputs, labels.unsqueeze(-1).to(images.device))
        rmse = torch.sqrt(mse)
        val_rmse += rmse.item()
        num_batches += images.shape[0]

avg_val_rmse = val_rmse / num_batches

print(f"Validation RMSE: {avg_val_rmse}")

Validation RMSE: 0.5668018499460188


In [ ]:
mse = nn.MSELoss()(x_preds, labels.unsqueeze(-1).to(images.device))
rmse = torch.sqrt(mse)
rmse

tensor(0.0756, device='cuda:0')

In [ ]:
df = pd.DataFrame({
    "y_pred": x_preds.cpu().numpy(),
    "y": labels.cpu().numpy(),
    'diff': labels.cpu().numpy() - x_preds.cpu().numpy()
})
df

,y_pred,y,diff
0,17.094698,17.01911,-0.075588


In [ ]:
mse = nn.MSELoss()(outputs.flatten(), labels.unsqueeze(-1).to(images.device))
rmse = torch.sqrt(mse)
rmse

tensor(17.0191, device='cuda:0')

In [ ]:
df = pd.DataFrame({
    "y_pred": outputs.flatten().cpu().numpy(),
    "y": labels.cpu().numpy(),
    'diff': labels.cpu().numpy() - outputs.flatten().cpu().numpy()
})
df

,y_pred,y,diff
0,0.0,17.01911,17.01911
